Copyright 2026 Google LLC

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a target="_blank" href="https://colab.research.google.com/github/lucianommartins/lab-sabadao/blob/main/examples/notebooks/05_agentic_quality_with_gemmaclaw.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# gbench agentic quality simulation with GemmaClaw

**Author:** [Luciano Martins](https://github.com/lucianommartins)

This notebook demonstrates how to evaluate multi-turn autonomous agent workflows using `gbench --quality-only` and the GemmaClaw / OpenClaw QA microservice (`48+ scenarios`). You will learn how `gbench` provisions the TypeScript simulation harness, manages `.buildstamp` compilation caching (~1–2 minutes initial compile), and tests session recall and tool routing against a local Ollama server.

## Learning objectives

1. Configure Ollama to serve a quantized Google Gemma 4 model (`unsloth/gemma-4-E4B-it-qat-GGUF`) with a context window of 8192 tokens.
2. Understand GemmaClaw microservice provisioning and TypeScript `.buildstamp` caching.
3. Execute multi-turn agentic simulation scenarios (`--quality-only`) over Ollama `/v1` endpoints.
4. Filter specific capability scenarios using `--scenarios memory/session_recall.json plugins/mcp_routing.json`.
5. Perform a clean session shutdown to terminate background servers and reclaim hardware memory.

## Useful resources

* [lab-sabadao GitHub repository](https://github.com/lucianommartins/lab-sabadao)
* [Ollama documentation](https://github.com/ollama/ollama)
* [OpenClaw agent simulation harness](https://github.com/google/openclaw)

## 1. Environment setup and installation

We clone the `lab-sabadao` repository from GitHub, change directory into the project root (`%cd lab-sabadao`), and install the package in editable mode (`%pip install -e .`). This builds and links the `gbench` CLI executable without installing unnecessary development linters.

In [ ]:
import os, sys
from pathlib import Path

# Safe environment setup: Always normalize to top-level repository
if Path("/content").exists():
    %cd -q /content
    if not Path("/content/lab-sabadao").is_dir():
        !git clone https://github.com/lucianommartins/lab-sabadao.git
    %cd -q /content/lab-sabadao
else:
    if not Path("pyproject.toml").is_file() and not Path("gbench").is_dir():
        if not Path("lab-sabadao").is_dir():
            !git clone https://github.com/lucianommartins/lab-sabadao.git
        %cd lab-sabadao

%pip install -e . -q
import gbench
print(f"gbench version {gbench.__version__} installed successfully.")

# Inspect available agent quality scenarios
!gbench --list quality

## 2. Installing Ollama locally

We check if the Ollama binary is present on the system. If it is not found, we install Ollama using its official Linux installation script (`curl -fsSL https://ollama.com/install.sh | sh`). Finally, we run `ollama --version` to verify that the installation succeeded and the CLI is available.

In [ ]:
import subprocess, os, shutil

if not shutil.which("ollama"):
    print("Installing Ollama locally...")
    # Ensure zstd is available (required by Ollama Linux tar.zst packages)
    subprocess.run("command -v zstd >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq zstd)", shell=True)
    # Run official Ollama installer
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
else:
    print("Ollama binary already installed.")

# Ensure binary directory is present in PATH for subsequent cells
for p in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]:
    if os.path.exists(os.path.join(p, "ollama")) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

!ollama --version

## 3. Launching background Ollama server

We launch the `ollama serve` process in the background and send a health check request to `http://localhost:11434/` to verify that the HTTP API is alive ("Ollama is running").

In [ ]:
import subprocess, time, requests
try:
    resp = requests.get("http://localhost:11434/", timeout=2)
    print("Ollama server already active:", resp.text.strip())
except Exception:
    print("Starting background ollama serve...")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    resp = requests.get("http://localhost:11434/")
    print("Server health check:", resp.text.strip())

## 4. Writing custom Modelfile for QAT model

We create an Ollama `Modelfile.qat` that configures our quantized Google Gemma 4 model (`hf.co/unsloth/gemma-4-E4B-it-qat-GGUF:latest`) with explicit parameters:
* **`num_ctx 8192`**: Context window of 8192 tokens.
* **`SYSTEM prompt`**: System instruction defining Gemma 4 AI assistant capabilities.

In [ ]:
HF_MODEL_ID = "hf.co/unsloth/gemma-4-E4B-it-qat-GGUF:UD-Q4_K_XL"
modelfile_content = f"""FROM {HF_MODEL_ID}
PARAMETER num_ctx 8192
SYSTEM "You are a helpful Gemma 4 AI assistant with reasoning, vision, and tool calling capabilities."
"""
with open("Modelfile.qat", "w", encoding="utf-8") as f:
    f.write(modelfile_content)
print("Created Modelfile.qat with valid Ollama parameters (num_ctx 8192, SYSTEM prompt).")

## 5. Registering model and running generation smoke test

We register our custom model tag (`gemma4-qat:4b`) using `ollama create -f Modelfile.qat`. This pulls the GGUF weights from Hugging Face Hub if not already cached. We then run a quick generation test (`ollama run`) to verify that the model loads into hardware memory and generates tokens correctly.

In [ ]:
import requests

MODEL_TAG = "gemma4-qat:4b"
print(f"Registering model {MODEL_TAG} from Modelfile.qat...")
!ollama create {MODEL_TAG} -f Modelfile.qat

print("Running quick generation smoke test via Ollama API (cold load into GPU VRAM)...")
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL_TAG, "prompt": "Reply with the single word: READY.", "stream": False},
    timeout=300,
)
print("Smoke test response:", resp.json().get("response", "").strip())

## 6. Verifying OpenAI REST endpoint readiness

Before launching `gbench`, we query `http://localhost:11434/v1/models` to verify that Ollama is serving standard OpenAI `/v1` REST payloads and that our registered model is listed.

In [ ]:
import requests
resp = requests.get("http://localhost:11434/v1/models")
print("OpenAI /v1/models endpoint HTTP status:", resp.status_code)
models = [m["id"] for m in resp.json().get("data", [])]
print("Available REST models:", models)

## 7. Listing and executing autonomous agent quality simulations

We can inspect all available agent quality simulation scenarios using `!gbench --list quality`.

We execute `gbench --quality-only` to evaluate multi-turn autonomous agent workflows. During initial invocation, `gbench` provisions the GemmaClaw simulation harness and compiles the `.buildstamp` cache. Once cached, subsequent runs execute immediately against the Ollama `/v1` endpoint.

In [ ]:
# List all agent quality scenarios
!gbench --list quality

# Execute autonomous agent quality evaluation
!gbench --quality-only \
        --models gemma4-qat:4b \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_quality

## 8. Filtering specific capability scenarios

You can filter the 48+ simulation scenarios using `--scenarios memory/session_recall.json plugins/mcp_routing.json` to focus on specific agent capabilities.

In [ ]:
!gbench --quality-only \
        --models gemma4-qat:4b \
        --scenarios memory/session_recall.json plugins/mcp_routing.json \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_quality_filtered

## 9. Analyzing multi-turn session recall and tool routing

We load `quality_results.json` to inspect how accurately the QAT model maintains long-context session recall across multiple agent turns and routes structured tool invocations.

In [ ]:
import json, glob, os
from pathlib import Path
import pandas as pd

def load_quality_results(results_base_dir):
    base = Path(results_base_dir)
    run_dirs = sorted([d for d in base.iterdir() if d.is_dir()], key=lambda d: d.stat().st_mtime, reverse=True) if base.exists() else []
    if not run_dirs:
        return pd.DataFrame()
    latest_dir = run_dirs[0]
    quality_files = sorted(latest_dir.glob("quality_*.json")) or sorted(latest_dir.glob("*.json"))
    records = []
    for qf in quality_files:
        with open(qf, "r", encoding="utf-8") as f:
            data = json.load(f)
        scenarios = data.get("scenarios", []) if isinstance(data, dict) else []
        for sc in scenarios:
            records.append({
                "scenario": sc.get("name", sc.get("scenario_id", "N/A")),
                "category": sc.get("category", "N/A"),
                "passed": "PASS" if sc.get("passed") else "FAIL",
                "score": sc.get("score", "N/A"),
                "tool_calls": sc.get("tool_calls_count", 0),
            })
    return pd.DataFrame(records)

print("=== Agent Quality Simulation Results ===")
df_quality = load_quality_results("./results_quality")
if not df_quality.empty:
    display(df_quality)
else:
    print("No agent quality results found.")

## 10. Session cleanup and server shutdown

We terminate background Ollama server processes and remove temporary Modelfiles.

In [ ]:
import subprocess, os

subprocess.run(["pkill", "-f", "ollama"], check=False)
if os.path.exists("Modelfile.qat"):
    os.remove("Modelfile.qat")
print("Session cleanup complete.")